# Metric Presentation and Visualization
## Necessary packages and functions call

- DDPM-TS: Interpretable Diffusion for Time Series Generation
- Metrics: 
    - discriminative_metrics
    - predictive_metrics
    - visualization

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
sys.path.append(os.path.join(os.path.dirname('__file__'), '../'))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

from Utils.metric_utils import display_scores
from Utils.discriminative_metric import discriminative_score_metrics
from Utils.predictive_metric import predictive_score_metrics

## Data Loading

Load original dataset and preprocess the loaded data.

In [3]:
iterations = 5
dataset_name = 'energy'
seq_length = 49
# ori_data = np.load('../toy_exp/samples/energy_ground_truth_24_train.npy')
# ori_data = np.load(f'../energy_results/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
ori_data = np.load(f'../energy_results/PANDORA_HALCOR/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')
fake_data = np.load('../energy_results/PANDORA_HALCOR/ddpm_fake_energy_0_to_1.npy')

## Evaluate the generated data

### 1. Discriminative score

To evaluate the classification accuracy between original and synthetic data using post-hoc RNN network. The output is | classification accuracy - 0.5 |.

- metric_iteration: the number of iterations for metric computation.

In [4]:
discriminative_score = []

for i in range(iterations):
    temp_disc, fake_acc, real_acc = discriminative_score_metrics(ori_data[:], fake_data[:ori_data.shape[0]])
    discriminative_score.append(temp_disc)
    print(f'Iter {i}: ', temp_disc, ',', fake_acc, ',', real_acc, '\n')
      
print('energy:')
display_scores(discriminative_score)
print()

Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Please use tf.global_variables instead.


training: 100%|██████████| 2000/2000 [00:21<00:00, 94.88it/s] 


Iter 0:  0.0805555555555556 , 0.7027777777777777 , 0.4583333333333333 



training: 100%|██████████| 2000/2000 [00:20<00:00, 99.07it/s] 


Iter 1:  0.1694444444444444 , 0.5361111111111111 , 0.8027777777777778 



...

Iter 2:  0.125 , 0.6888888888888889 , 0.5611111111111111 



training: 100%|██████████| 2000/2000 [00:20<00:00, 99.60it/s] 


Iter 3:  0.12638888888888888 , 0.5638888888888889 , 0.6888888888888889 



training: 100%|██████████| 2000/2000 [00:20<00:00, 96.38it/s] 

Iter 4:  0.14583333333333337 , 0.5222222222222223 , 0.7694444444444445 

energy:
Final Score:  0.12944444444444445 ± 0.04065292427043249



## Evaluate the generated data

### 2. Predictive score

To evaluate the prediction performance on train on synthetic, test on real setting. More specifically, we use Post-hoc RNN architecture to predict one-step ahead and report the performance in terms of MAE. 

The model learns to predict the last dimension with one more step.

In [5]:
predictive_score = []
for i in range(iterations):
    temp_pred = predictive_score_metrics(ori_data, fake_data[:ori_data.shape[0]])
    predictive_score.append(temp_pred)
    print(i, ' epoch: ', temp_pred, '\n')
      
print('energy:')
display_scores(predictive_score)
print()

training: 100%|██████████| 5000/5000 [00:32<00:00, 154.04it/s]


0  epoch:  0.05069328274805577 



training: 100%|██████████| 5000/5000 [00:34<00:00, 146.64it/s]


1  epoch:  0.04990116566685832 



training: 100%|██████████| 5000/5000 [00:32<00:00, 154.94it/s]


2  epoch:  0.049073756912195135 



training: 100%|██████████| 5000/5000 [00:30<00:00, 162.59it/s]


3  epoch:  0.0491345207869232 



training: 100%|██████████| 5000/5000 [00:31<00:00, 159.74it/s]


4  epoch:  0.050841617175663996 

energy:
Final Score:  0.04992886865793929 ± 0.0010352102912489826



In [6]:
import os
import pandas as pd
import numpy as np
import scipy.stats

def compute_score(results, confidence=0.95):
    """Return mean ± CI string like display_scores."""
    mean = np.mean(results)
    sigma = scipy.stats.sem(results)
    sigma = sigma * scipy.stats.t.ppf((1 + confidence) / 2., len(results)-1)
    return f"{mean:.4f} ± {sigma:.4f}"

# Example usage after your iterations
disc_result = compute_score(discriminative_score)
pred_result = compute_score(predictive_score)

results = {
    "Metric": ["Discriminative", "Predictive"],
    "Result": [disc_result, pred_result]
}

results_df = pd.DataFrame(results)

save_folder = "../figures/"
os.makedirs(save_folder, exist_ok=True)

save_path = os.path.join(save_folder, f"PANDORA_HALCOR_{dataset_name}_metrics.csv")
results_df.to_csv(save_path, index=False)

print(f"✅ Compact metrics saved to {save_path}")

✅ Compact metrics saved to ../figures/PANDORA_HALCOR_energy_metrics.csv
